### Respostas às Perguntas-Guia para Reflexão (Semana 2)

**1. Por que a condição é $\gcd(\det A, 26) = 1$ e não simplesmente $\det A \neq 0$?** 

Na álgebra linear clássica (construída sobre o corpo dos números reais $\mathbb{R}$), basta que o determinante seja diferente de zero ($\det A \neq 0$) para que a matriz seja invertível, pois todo número real não nulo possui um inverso multiplicativo. 

No entanto, na Cifra de Hill, a matriz $A$ possui entradas no anel de inteiros módulo 26, denotado por $\mathbb{Z}_{26}$. Para que possamos decifrar a mensagem, precisamos encontrar a matriz inversa $A^{-1}$, cujo cálculo depende do inverso multiplicativo do determinante:

$$A^{-1} = (\det A)^{-1} \cdot \text{adj}(A) \pmod{26}$$

A congruência linear $x \cdot \det A \equiv 1 \pmod{26}$ só possui solução se, e somente se, o determinante for coprimo em relação à base modular. Como a fatoração em primos de $26$ é $2 \times 13$, a condição matemática necessária e suficiente para a existência da inversa é:

$$\gcd(\det A, 26) = 1$$

Se tivéssemos $\det A = 2$, por exemplo, teríamos $\det A \neq 0$, mas a matriz continuaria não invertível em $\mathbb{Z}_{26}$.


**2. Por que o ataque por texto claro conhecido funciona aqui mas não funciona contra AES?** 

O ataque por texto claro conhecido é devastador contra a Cifra de Hill porque o processo de cifragem é uma transformação **estritamente linear**. Se um atacante obtiver uma matriz de textos claros $P$ e sua correspondente matriz de textos cifrados $C$ com blocos linearmente independentes suficientes, ele pode modelar a interceptação como um sistema linear simples:

$$C \equiv P \cdot A \pmod{26}$$

Se a matriz de textos claros $P$ for invertível, o atacante recupera a chave secreta $A$ trivialmente com uma simples multiplicação de matrizes:

$$A \equiv P^{-1} \cdot C \pmod{26}$$

Em contraste, o **AES (Advanced Encryption Standard)** é resistente a este ataque porque introduz operações **não-lineares** em sua arquitetura (as chamadas redes de Substituição-Permutação). A etapa de *SubBytes* (S-box) utiliza o inverso multiplicativo sobre o corpo finito $\mathbb{F}_{2^8}$ somado a uma transformação afim. Essa não-linearidade impede que o AES seja expresso como um sistema de equações lineares simples, bloqueando completamente a resolução algébrica direta.


**3. Se trocássemos o alfabeto de 26 letras por um alfabeto de 256 bytes, o que mudaria?** 

Essa mudança teria duas implicações profundas:

* **Implicação Prática:** A cifra deixaria de operar apenas sobre texto legível (caracteres de 'A' a 'Z') e passaria a operar sobre qualquer tipo de dado binário arbitrário, já que cada bloco processaria diretamente os valores hexadecimais dos bytes ($0$ a $255$). Isso permitiria cifrar imagens, executáveis e arquivos ZIP.
* **Implicação Matemática:** A álgebra linear deixaria de ser feita sobre o anel $\mathbb{Z}_{26}$ e passaria a operar sobre o anel $\mathbb{Z}_{256}$. Como $256 = 2^8$, os únicos divisores do módulo são as potências de 2. Logo, a nova condição de invertibilidade da chave seria:
    $$\gcd(\det A, 256) = 1$$
    Na prática, isso significa que **qualquer matriz cujo determinante resulte em um número ímpar** seria invertível em $\mathbb{Z}_{256}$. 

*(Nota para o relatório: O AES resolve isso de maneira muito mais elegante operando em um corpo finito $\mathbb{F}_{2^8}$, garantindo que todo elemento não nulo possua um inverso, diferentemente do anel $\mathbb{Z}_{256}$).*

In [8]:
import numpy as np
import math

# ==========================================
# UTILITÁRIOS MATEMÁTICOS
# ==========================================

def inverso_modular(a, m):
    """Encontra o inverso modular de a mod m usando força bruta simples ou Euclides."""
    a = a % m
    for x in range(1, m):
        if (a * x) % m == 1:
            return x
    raise ValueError(f"Não existe inverso modular para {a} mod {m}")

def cofator(matriz, linha, coluna):
    """Calcula o cofator de uma matriz para encontrar a matriz adjunta."""
    menor = np.delete(np.delete(matriz, linha, axis=0), coluna, axis=1)
    return ((-1) ** (linha + coluna)) * int(round(np.linalg.det(menor)))

def inversa_modular_matriz(matriz, modulo=26):
    """
    Calcula a matriz inversa em Z_26 usando a fórmula da matriz adjunta.
    Fonte: Roteiro da Semana 2.
    """
    n = matriz.shape[0]
    det = int(round(np.linalg.det(matriz))) % modulo
    
    # Verifica a condição gcd(det A, 26) = 1
    if math.gcd(det, modulo) != 1:
        raise ValueError("A matriz não é invertível sobre Z_26 (gcd(det, 26) != 1).")
    
    det_inv = inverso_modular(det, modulo)
    
    # Calcula a matriz adjunta (transposta da matriz de cofatores)
    adjunta = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            adjunta[j, i] = cofator(matriz, i, j) % modulo
            
    # Multiplica a adjunta pelo inverso do determinante
    matriz_inversa = (det_inv * adjunta) % modulo
    return matriz_inversa

# ==========================================
# CONVERSÃO DE TEXTO
# ==========================================

def texto_para_vetores(texto, n):
    """Codifica o texto claro em vetores numéricos e aplica padding."""
    texto = ''.join(c.upper() for c in texto if c.isalpha())
    while len(texto) % n != 0:
        texto += 'X'
    
    numeros = [ord(c) - ord('A') for c in texto]
    vetores = np.array(numeros).reshape(-1, n)
    return vetores

def vetores_para_texto(vetores):
    """Converte vetores numéricos de volta para texto."""
    numeros = vetores.flatten()
    return ''.join(chr(int(num) + ord('A')) for num in numeros)

# ==========================================
# CIFRAGEM E DECIFRAGEM
# ==========================================

def cifrar_hill(texto_claro, chave_matriz):
    """Cifra o texto por blocos usando a matriz chave."""
    n = chave_matriz.shape[0]
    vetores = texto_para_vetores(texto_claro, n)
    
    vetores_cifrados = []
    for vetor in vetores:
        # Multiplicação matriz-vetor em Z_26
        vetor_cifrado = np.dot(chave_matriz, vetor) % 26
        vetores_cifrados.append(vetor_cifrado)
        
    return vetores_para_texto(np.array(vetores_cifrados))

def decifrar_hill(texto_cifrado, chave_matriz):
    """Decifra o texto revertendo a operação com a inversa_modular_matriz."""
    n = chave_matriz.shape[0]
    chave_inversa = inversa_modular_matriz(chave_matriz, 26)
    vetores_cifrados = texto_para_vetores(texto_cifrado, n)
    
    vetores_claros = []
    for vetor in vetores_cifrados:
        vetor_claro = np.dot(chave_inversa, vetor) % 26
        vetores_claros.append(vetor_claro)
        
    return vetores_para_texto(np.array(vetores_claros))

# ==========================================
# VALIDAÇÃO DO ALGORITMO
# ==========================================
if __name__ == "__main__":
    # 1. Definição da Chave (n=3)
    chave = np.array([
        [6, 24, 1],
        [13, 16, 10],
        [20, 17, 15]
    ])
    
    # 2. Cifragem
    mensagem_original = "MATEMATICA"
    print(f"Mensagem Original: {mensagem_original}")
    
    cifrado = cifrar_hill(mensagem_original, chave)
    print(f"Texto Cifrado:     {cifrado}")
    
    # 3. Decifragem
    decifrado = decifrar_hill(cifrado, chave)
    print(f"Texto Decifrado:   {decifrado}")
    
    assert mensagem_original in decifrado, "Falha na validação de cifragem/decifragem!"

Mensagem Original: MATEMATICA
Texto Cifrado:     NIFAKYWFADAI
Texto Decifrado:   MATEMATICAXX


In [10]:
# ==========================================
# ATAQUE POR TEXTO CLARO CONHECIDO
# ==========================================

def ataque_texto_claro_conhecido(texto_claro, texto_cifrado, n):
    """
    Demonstra o ataque recuperando a chave a partir de um par claro/cifrado.
    Requer pelo menos 'n' blocos linearmente independentes.
    """
    # Converte os primeiros 'n' blocos (n*n letras) em matrizes colunas
    vetores_p = texto_para_vetores(texto_claro[:n*n], n)
    vetores_c = texto_para_vetores(texto_cifrado[:n*n], n)
    
    # Transpõe para que os vetores virem colunas (matriz P e C são n x n)
    P = vetores_p.T
    C = vetores_c.T
    
    try:
        # Tenta calcular a inversa de P em Z_26
        P_inv = inversa_modular_matriz(P, 26)
    except ValueError:
        return "Ataque falhou: Os blocos de texto claro escolhidos não formam uma matriz invertível em Z_26. Tente usar uma porção diferente do texto."
    
    # Calcula a Chave: A = C * P^-1 mod 26
    chave_recuperada = np.dot(C, P_inv) % 26
    return chave_recuperada

# Testando o ataque
if __name__ == "__main__":
    print("\n--- DEMONSTRAÇÃO DO ATAQUE ---")
    
    # Trocamos para uma palavra que forma uma matriz P invertível em Z_26
    texto_conhecido = "UNIVERSIDADE" 
    
    texto_interceptado = cifrar_hill(texto_conhecido, chave)
    
    print(f"Interceptado Claro:   {texto_conhecido}")
    print(f"Interceptado Cifrado: {texto_interceptado}")
    
    # O ataque vai usar as 9 primeiras letras (UNI VER SID)
    chave_descoberta = ataque_texto_claro_conhecido(texto_conhecido, texto_interceptado, 3)
    
    print("\nChave Original:")
    print(chave)
    print("\nChave Recuperada pelo Ataque:")
    print(chave_descoberta)
    
    if isinstance(chave_descoberta, np.ndarray) and np.array_equal(chave, chave_descoberta):
        print("\nSucesso! A chave secreta foi totalmente comprometida.")


--- DEMONSTRAÇÃO DO ATAQUE ---
Interceptado Claro:   UNIVERSIDADE
Interceptado Cifrado: YCNFNPRCVYKH

Chave Original:
[[ 6 24  1]
 [13 16 10]
 [20 17 15]]

Chave Recuperada pelo Ataque:
[[ 6 24  1]
 [13 16 10]
 [20 17 15]]

Sucesso! A chave secreta foi totalmente comprometida.


**Análise Crítica: A Cifra de Hill**

**Matemática Essencial**
A Cifra de Hill, introduzida por Lester S. Hill em 1929, representa um marco na transição da criptografia clássica, baseada em substituições simples, para uma abordagem genuinamente algébrica. A cifra opera utilizando álgebra linear sobre o anel de inteiros módulo 26 ($\mathbb{Z}_{26}$), representando o alfabeto padrão. O processo de cifragem consiste em dividir a mensagem original em vetores de tamanho $n$ e multiplicá-los por uma matriz chave $A \in M_{n}(\mathbb{Z}_{26})$. Para que a mensagem seja recuperável, a transformação precisa ser bijetora, o que impõe uma condição matemática estrita sobre a matriz chave: ela deve ser invertível. Como estamos operando sobre um anel modular onde 26 não é primo, a condição de invertibilidade exige que o determinante da matriz seja coprimo à base. 

Formalmente, $\gcd(\det A, 26) = 1$. 

O processo de decifragem, portanto, consiste na multiplicação do texto cifrado pela matriz inversa modular 

$A^{-1} \equiv (\det A)^{-1} \cdot \text{adj}(A) \pmod{26}$.

**Vantagens: Difusão e Resistência Básica**
O principal mérito da Cifra de Hill em relação às cifras clássicas (como César ou Vigenère) é a introdução do conceito de difusão de Shannon. Como a cifragem ocorre por blocos, a alteração de um único caractere no texto claro modifica completamente todos os caracteres do bloco cifrado correspondente. Isso mascara as frequências individuais das letras, tornando a cifra imune a análises de frequência estáticas ingênuas. Letras idênticas no texto claro frequentemente resultarão em caracteres distintos no texto cifrado, dependendo de sua posição no vetor.

**Desvantagens: O Calcanhar de Aquiles da Linearidade**
Apesar de sua elegância matemática, a Cifra de Hill possui uma vulnerabilidade fatal: sua estrita linearidade. Todo o processo criptográfico pode ser reduzido a um sistema de equações lineares simples ($C \equiv P \cdot A \pmod{26}$). Se um adversário tiver acesso a pares de texto claro e texto cifrado (Ataque por Texto Claro Conhecido) cujos vetores formem matrizes invertíveis, ele pode recuperar a matriz chave secretamente realizando uma simples multiplicação de matrizes: $A \equiv C \cdot P^{-1} \pmod{26}$. A ausência de componentes não-lineares impede que a Cifra de Hill mascare a estrutura algébrica da transformação.

**Uso Prático e Legado**
Devido à sua suscetibilidade a ataques algébricos diretos, a Cifra de Hill não é, e não deve ser, utilizada em sistemas de produção modernos. No entanto, seu valor didático é inestimável. Ela ilustra perfeitamente por que a linearidade é um defeito em criptografia e justifica o design de cifras simétricas modernas como o AES. O AES, embora opere sobre blocos e matrizes, resolve essa falha crítica substituindo o anel $\mathbb{Z}_{26}$ pelo corpo finito $\mathbb{F}_{2^8}$  e, fundamentalmente, introduzindo a operação de *SubBytes* (S-box) , uma transformação não-linear baseada em inverso multiplicativo que quebra a previsibilidade explorada nos ataques contra a Hill.